## 介紹 

本課程將涵蓋： 
- 甚麼是函式呼叫及其使用場景 
- 如何使用 OpenAI 建立函式呼叫 
- 如何將函式呼叫整合入應用程式 

## 學習目標 

完成本課程後，您將了解如何並明白： 

- 使用函式呼叫的目的 
- 使用 OpenAI 服務設定函式呼叫 
- 為您的應用程式使用情境設計有效的函式呼叫 


## 了解函式呼叫

在本課程中，我們想為我們的教育創業公司建立一個功能，讓用戶能使用聊天機械人來尋找技術課程。我們將根據他們的技能水平、當前職位和感興趣的技術推薦課程。

為完成此目標，我們將結合使用：
 - `OpenAI` 來為用戶創建聊天體驗
 - `Microsoft Learn Catalog API` 根據用戶的要求幫助他們找到課程
 - `函式呼叫` 將用戶的查詢傳送給函式以進行 API 請求。

要開始，讓我們先看看為何我們首先想要使用函式呼叫：


### 為何使用 Function Calling 

如果你已經完成這個課程中的其它課程，你可能已經體會到使用大型語言模型 (LLMs) 的威力。希望你也能看到它們的一些限制。 

Function Calling 是 OpenAI 服務的一項功能，旨在解決以下挑戰：

回應格式不一致：
- 在使用 function calling 之前，大型語言模型的回應是非結構化且不一致的。開發者必須撰寫複雜的驗證程式碼以處理輸出中的各種變化。

與外部數據整合有限：
- 在此功能之前，將來自應用其他部分的數據整合到聊天上下文中是困難的。

通過標準化回應格式並實現與外部數據的無縫整合，function calling 簡化了開發流程，並減少了額外驗證邏輯的需求。

用戶無法獲得如「斯德哥爾摩目前的天氣如何？」的答案。這是因為模型的知識受限於其訓練數據的時間範圍。 

讓我們來看下面說明這個問題的範例： 

假設我們想建立一個學生資料庫，以便建議適合他們的課程。以下有兩個學生的描述，它們在所含資料上非常相似。


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


我們想將這個發送給 LLM 來解析數據。稍後可以在我們的應用程式中使用這個來發送到 API 或儲存在資料庫中。

讓我們建立兩個相同的提示，指示 LLM 我們感興趣的資訊是哪些：


我哋想將呢啲嘢發送畀一個大型語言模型 (LLM)，嚟分析對我哋產品重要嘅部分。所以我哋可以創建兩個相同嘅提示詞，嚟指示呢個大型語言模型： 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


建立這兩個提示後，我們將使用 `client.responses.create` 將它們發送到 LLM。我們將提示存儲在 `input` 變數中，並將角色分配為 `user`。這是為了模擬來自使用者向聊天機械人發送消息的情況。



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

deployment="gpt-4o-mini"


: 

現在我們可以同時向 LLM 發送兩個請求，並檢查我們收到的回應。 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



即使提示相同且描述相似，我們也可能得到不同格式的 `Grades` 屬性。

如果多次運行上述儲存格，格式可能是 `3.7` 或 `3.7 GPA`。

這是因為 LLM 接收以書面提示形式的非結構化數據，並且也返回非結構化數據。我們需要一個結構化的格式，以便在存儲或使用這些數據時知道會得到什麼。

通過使用功能調用，我們可以確保收到結構化的數據。使用功能調用時，LLM 並不真正調用或執行任何函數。取而代之的是，我們為 LLM 創建了一個它用來回答的結構。然後，我們使用這些結構化的回答來決定在我們的應用中運行哪個函數。
 


![函數調用流程圖](../../../../translated_images/zh-MO/Function-Flow.083875364af4f4bb.webp)


然後我們可以把從函數返回的結果傳回給 LLM。LLM 之後會用自然語言回應來回答用戶的查詢。


### 使用函數調用的應用場景 

<strong>調用外部工具</strong> 
聊天機器人在提供用戶問題答案方面非常出色。通過使用函數調用，聊天機器人可以利用用戶的訊息來完成特定任務。例如，學生可以要求聊天機器人「發送電子郵件給我的老師，說我需要更多這門課的協助」。這可以調用一個名為 `send_email(to: string, body: string)` 的函數


**創建 API 或資料庫查詢**
用戶可以使用自然語言來查找資訊，並將其轉換為格式化的查詢或 API 請求。例如，一位老師請求「誰完成了最後一個作業」，就可以調用一個名為 `get_completed(student_name: string, assignment: int, current_status: string)` 的函數


<strong>創建結構化數據</strong>
用戶可以使用 LLM 從一段文字或 CSV 中提取重要資訊。例如，學生可以將一篇關於和平協議的維基百科文章轉換成 AI 問題卡片。這可以通過調用一個名為 `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` 的函數來完成


## 2. 創建您的第一個函數調用 

創建函數調用的過程包含三個主要步驟： 
1. 使用您的函數列表和用戶訊息呼叫 Chat Completions API 
2. 閱讀模型的回應以執行操作，即執行函數或 API 呼叫 
3. 使用函數的回應進行另一個 Chat Completions API 呼叫，以利用該資訊創建對用戶的回應。 


![函數呼叫流程](../../../../translated_images/zh-MO/LLM-Flow.3285ed8caf4796d7.webp)


### 函式呼叫的元素 

#### 用戶輸入 

第一步是建立用戶訊息。這可以透過取得文字輸入的值來動態指派，或者你也可以在這裡指派一個值。如果這是你第一次使用 Chat Completions API，我們需要定義訊息的 `role` 和 `content`。 

`role` 可以是 `system`（建立規則）、`assistant`（模型）或 `user`（最終用戶）。對於函式呼叫，我們會將其指派為 `user` 並加入示範問題。 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### 建立函式。

接下來我們會定義一個函式及該函式的參數。我們這裡只會使用一個名為 `search_courses` 的函式，但你可以建立多個函式。

<strong>重要</strong> ：函式會包含在傳送給大型語言模型的系統訊息中，並且會計入你可用的代幣數量裡。


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


<strong>定義</strong> 

函數定義結構有多層級，每層都有其屬性。以下是巢狀結構的細分說明：

**頂層函數屬性：**

`name` - 我們希望呼叫的函數名稱。 

`description` - 這是函數運作方式的描述。在此特別需要具體清晰。 

`parameters` - 一份希望模型在回應中產生的數值及格式清單。 

**參數物件屬性：**

`type` - 參數物件的資料類型（通常是 "object"）

`properties` - 模型將用來產生回應的特定數值清單。 

**個別參數屬性：**

`name` - 由屬性鍵隱式定義（例如 "role"、"product"、"level"）

`type` - 該特定參數的資料類型（例如 "string"、"number"、"boolean"） 

`description` - 該特定參數的描述 

**選用屬性：**

`required` - 列出必須的參數的陣列，以完成函數呼叫。 


### 呼叫函數
定義函數後，我們現在需要在呼叫 Chat Completion API 時包含該函數。我們透過在請求中加入 `functions` 來做到這點。在這個案例裡，`functions=functions`。

也有一個選項可以將 `function_call` 設定為 `auto`。這表示我們會讓大型語言模型根據使用者訊息決定應該呼叫哪個函數，而不是自己指定。


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


現在讓我們來看看回應，了解它的格式： 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

你可以看到函數名稱被呼叫，並且從使用者訊息中，語言模型能找到符合函數參數的資料。 


## 3.將功能調用整合到應用程式中。


在測試了來自大型語言模型格式化的回應後，現在我們可以將其整合到應用程式中。

### 管理流程

為了將這整合到我們的應用程式中，讓我們採取以下步驟：

首先，我們先呼叫 OpenAI 服務，並將訊息存到一個叫做 `response_message` 的變數中。


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


現在我們將定義一個函數，用來調用 Microsoft Learn API 以獲取課程列表： 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



作為最佳實務，我們會先檢查模型是否想要呼叫一個函數。之後，我們會建立一個可用的函數並將其與被呼叫的函數相匹配。
接著，我們會將該函數的參數對應到來自LLM的參數。

最後，我們會附加函數呼叫訊息及由 `search_courses` 訊息返回的數值。這讓LLM擁有回應使用者所需的所有資訊，並使用自然語言進行回應。


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


現在我們將發送更新後的訊息給 LLM，這樣我們就能收到自然語言回應，而不是 API JSON 格式的回應。


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Code Challenge 

Great work! To continue your learning of OpenAI Function Calling you can build: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - More parameters of the function that might help learners find more courses. You can find the available API parameters here: 
 - Create another function call that takes more information from the learner like their native language 
 - Create error handling when the function call and/or API call does not return any suitable courses 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
